In [ ]:
!pip install tensorflow scikit-learn pandas numpy -q

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Downloading MAGE...")
ds = load_dataset("yaful/MAGE")
df = ds['train'].to_pandas()

print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print(f"Labels  :\n{df['label'].value_counts()}")

In [ ]:
# label: 0 = AI, 1 = Human
human_df = df[df['label'] == 1][['text']].copy()
human_df['label'] = 0  # remap to your convention: 0=human

ai_df = df[df['label'] == 0][['text']].copy()
ai_df['label'] = 1     # remap: 1=AI

print(f"Available human : {len(human_df)}")
print(f"Available AI    : {len(ai_df)}")

n = min(25000, len(human_df), len(ai_df))
balanced = pd.concat([
    human_df.sample(n, random_state=42),
    ai_df.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

balanced = balanced[['text', 'label']]
print(f"\nTotal : {len(balanced)}")

In [ ]:
print("=== MAGE Balance Check ===")
print(balanced['label'].value_counts())
print(f"\nHuman : {(balanced['label']==0).sum()}")
print(f"AI    : {(balanced['label']==1).sum()}")
print(f"Ratio : {(balanced['label']==0).sum() / (balanced['label']==1).sum():.2f}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    balanced,
    test_size=0.2,
    random_state=42,
    stratify=balanced["label"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
def tokenize(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df["text"])
val_encodings = tokenize(val_df["text"])
test_encodings = tokenize(test_df["text"])

print("Tokenization complete!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

In [ ]:
train_dataset = TextDataset(
    train_encodings,
    train_df["label"]
)

val_dataset = TextDataset(
    val_encodings,
    val_df["label"]
)

test_dataset = TextDataset(
    test_encodings,
    test_df["label"]
)

print("Datasets created!")

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("DistilBERT loaded!")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_dir="./distilbert_logs",

    load_best_model_at_end=True,

    fp16=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Trainer ready!")

In [ ]:
trainer.train()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

import numpy as np

predictions = trainer.predict(
    test_dataset
)

preds = np.argmax(
    predictions.predictions,
    axis=1
)

labels = predictions.label_ids

print(
    "Accuracy:",
    accuracy_score(labels, preds)
)

print(
    classification_report(
        labels,
        preds,
        target_names=[
            "Human",
            "AI Generated"
        ]
    )
)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

save_path = (
    "/content/drive/MyDrive/"
    "distilbert_hc3plus_model"
)

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("Model saved!")

In [ ]:
import torch

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification
)

model_path = (
    "/content/drive/MyDrive/"
    "distilbert_hc3plus_model"
)

tokenizer = DistilBertTokenizer.from_pretrained(
    model_path
)

model = DistilBertForSequenceClassification.from_pretrained(
    model_path
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

print(
    f"Model loaded on {device}"
)

In [ ]:
def predict_text(text):

    if len(text.split()) < 10:
        print(
            "⚠️ Please enter at least 10 words."
        )
        return

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

        confidence = torch.softmax(
            outputs.logits,
            dim=1
        ).max().item()

    label = (
        "🤖 AI Generated"
        if prediction == 1
        else "👤 Human Written"
    )

    print("Prediction:", label)
    print(
        f"Confidence: "
        f"{confidence*100:.2f}%"
    )

In [ ]:
while True:

    text = input(
        "\nEnter text (or type quit): "
    )

    if text.lower() == "quit":
        print("Exiting...")
        break

    predict_text(text)